In [ ]:
import numpy as np
import ifcopenshell
import heapq

# for some reason ifcopenshell.api is not imported when importing ifcopenshell, so we need to import it explicitly
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.open("Function_Model_v3.ifc")

# defined as a dictionary, because there can be multiple boundaries related by openings
def get_boundary_room(space):
    b_room = {}
    for boundary in space.BoundedBy:
        corr_boundary = boundary.CorrespondingBoundary
        b_room[boundary.RelatedBuildingElement.Name] = corr_boundary.RelatingSpace
    return b_room

def get_openings(space):
    openings = {}
    for boundary in space.BoundedBy:
        if boundary.RelatedBuildingElement.is_a("IfcOpeningElement"):
            openings[boundary.RelatedBuildingElement.Name] = boundary.RelatedBuildingElement
    return openings

def get_coordinates(entity):
    matrix = ifcopenshell.util.placement.get_local_placement(entity.ObjectPlacement)
    return matrix[:,3][0:3]

###### other functions to add later:
# get length/width of a space
# calculate distance between spaces




Do the edge weightings need to be known before implementing the algorithm?
Probably not, but
since I'm finding the worst case from a given building, there is no "start". So I guess will need to iterate through every path anyway?


In [2]:
import networkx as nx
import plotly.graph_objects as go

def build_nx_space_graph(model):
    """Build a graph of IfcSpaces connected through IfcOpeningElements."""
    G = nx.Graph()

    for space in model.by_type("IfcSpace"):
        s1 = space.Name or f"IfcSpace_{space.id()}"
        G.add_node(s1, ifc_id=space.id())

        for b in getattr(space, "BoundedBy", []):
            if not b.is_a("IfcRelSpaceBoundary2ndLevel"):
                continue

            cb = getattr(b, "CorrespondingBoundary", None)
            if not cb:
                continue

            elem = getattr(b, "RelatedBuildingElement", None)
            if not elem or not elem.is_a("IfcOpeningElement"):
                continue

            s2_obj = cb.RelatingSpace
            s2 = s2_obj.Name or f"IfcSpace_{s2_obj.id()}"
            opening_name = elem.Name or f"IfcOpeningElement_{elem.id()}"

            if not G.has_edge(s1, s2):
                G.add_edge(
                    s1,
                    s2,
                    opening=opening_name,
                    opening_id=elem.id(),
                    boundary_id=b.id(),
                    corresponding_boundary_id=cb.id(),
                )

    return G

G = build_nx_space_graph(model)

In [3]:

# perhaps just try to make this with one goal first (one exit), then we can try to make it with multiple goals (multiple exits) later.

def h_n(current, goal):
    coords1 = get_coordinates(current)[0:2]
    coords2 = get_coordinates(goal)[0:2]
    distance = np.linalg.norm(coords1 - coords2)
    return distance


# A* algorithm. To be used for multiple starting points and goals, so we can find the shortest path to an exit in the building.
def a_star(start, goal, h):
    open_set = set()
    closed_set = set()

    open_set.add(start)
                          
    g_score = {start: 0}                                    
    f_score = {start: h(start)}                         
    while open_set is not empty: 
        current = node in open_set with lowest f_score 
        if current == goal: 
            return reconstruct_path(current) 
        open_set.remove(current) 
        closed_set.add(current) 
        for neighbor in get_neighbors(current): 
            if neighbor in closed_set: 
                continue 

            # change cost to weights i.e. distance to neighbours i think
            tentative_g = g_score[current] + cost(current, neighbor) 


            if neighbor not in open_set: 
                open_set.add(neighbor) 
            elif tentative_g >= g_score[neighbor]: 
                continue 
            g_score[neighbor] = tentative_g 
            f_score[neighbor] = g_score[neighbor] + h(neighbor)

SyntaxError: invalid syntax (598685760.py, line 20)

In [ ]:
print(type(G.nodes))
space_by_node = {
    space.Name or f"IfcSpace_{space.id()}": space
    for space in model.by_type("IfcSpace")
}

print(space_by_node)

<class 'networkx.classes.reportviews.NodeView'>


TypeError: 'dict' object is not callable

In [22]:
import heapq
import networkx as nx

# Map graph node names to their IFC space entities
space_by_node = {
    space.Name or f"IfcSpace_{space.id()}": space
    for space in model.by_type("IfcSpace")
}


def node_coordinates(node):
    """Return the x, y coordinates for a graph node."""
    return get_coordinates(space_by_node[node])[:2]


def edge_cost(graph, current, neighbor):
    """Use Euclidean distance between connected spaces as the edge weight."""
    return float(np.linalg.norm(
        node_coordinates(current) - node_coordinates(neighbor)
    ))


def h_n(current, goal):
    """Euclidean heuristic."""
    return float(np.linalg.norm(
        node_coordinates(current) - node_coordinates(goal)
    ))


def a_star(graph, start, goal):
    """Find the shortest path between two graph nodes using A*."""
    if start not in graph:
        raise ValueError(f"Unknown start node: {start}")

    if goal not in graph:
        raise ValueError(f"Unknown goal node: {goal}")

    # Priority queue entries are: (estimated_total_cost, path_cost, node)
    open_set = [(h_n(start, goal), 0.0, start)]

    # Cheapest known cost from start to each node
    g_score = {start: 0.0}

    # Used to reconstruct the path
    came_from = {}

    while open_set:
        _, current_cost, current = heapq.heappop(open_set)

        # Ignore outdated queue entries
        if current_cost > g_score.get(current, float("inf")):
            continue

        if current == goal:
            path = [current]

            while current in came_from:
                current = came_from[current]
                path.append(current)

            path.reverse()
            return path, g_score[goal]
        # Examine every space connected to the current space.
        for neighbor in graph.neighbors(current):
            cost = edge_cost(graph, current, neighbor)
            tentative_g_score = g_score[current] + cost

            # Calculate the cost of the route through the current node.
            if tentative_g_score < g_score.get(neighbor, float("inf")):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score

                # f(n) = g(n) + h(n)
                # g(n): cost from start to this node
                # h(n): estimated cost from this node to the goal

                f_score = tentative_g_score + h_n(neighbor, goal)
                heapq.heappush(
                    open_set,
                    (f_score, tentative_g_score, neighbor)
                )

        # for debugging purposes, print the current cost of the node being processed
        # print(current_cost)
        
    raise nx.NetworkXNoPath(
        f"No path exists between {start!r} and {goal!r}"
    )

In [26]:
# start = list(G.nodes)[0]
start = "north_clean"
goal1 = "ne_lobby"
goal2 = "entry_lobby"
goal3 = "s2_office"
# note: the goal is currently set to the lobby, but it should be set to the exit door of these rooms

for room in list(G.nodes):
    path, distance = a_star(G, room, goal1)

    # print("Path:", path)
    print("Distance:", distance)


for room in list(G.nodes):
    path, distance = a_star(G, room, goal2)

    # print("Path:", path)
    print("Distance:", distance)

for room in list(G.nodes):
    path, distance = a_star(G, room, goal3)

    # print("Path:", path)
    print("Distance:", distance)

Distance: 24.316037017023
Distance: 16.094115100585213
Distance: 21.110089582178993
Distance: 22.323079701544188
Distance: 11.523808536739399
Distance: 6.794115100585211
Distance: 13.815510973201995
Distance: 21.87425350723258
Distance: 21.81425495901525
Distance: 11.42120922108892
Distance: 0.0
Distance: 25.574253507232584
Distance: 26.641262313474115
Distance: 31.468269667932987
Distance: 0.0
Distance: 8.221921916437786
Distance: 13.237896398031566
Distance: 14.45088651739676
Distance: 22.251615352591973
Distance: 17.521921916437787
Distance: 24.54331778905457
Distance: 14.002060323085155
Distance: 13.942061774867824
Distance: 22.149016036941497
Distance: 24.316037017023
Distance: 17.702060323085156
Distance: 18.76906912932669
Distance: 23.59607648378556
Distance: 14.45088651739676
Distance: 6.228964600958975
Distance: 11.244939082552756
Distance: 0.0
Distance: 20.25865803711316
Distance: 15.528964600958975
Distance: 22.550360473575758
Distance: 12.009103007606345
Distance: 11.949104

In [13]:
print(G.nodes)

goal = list(G.nodes)[-1]

print(goal)

['entry_lobby', 'west_corridor', 's1_office', 's2_office', 's3_office', 'east_corridor', 'east_lobby', 'n1_office', 'north_corridor', 'n3_office', 'ne_lobby', 'n2_office', 'north_clean', 'north_store']
north_store
